# Hugging Face Audio Course — Unit 2: A gentle introduction to audio applications

Using the 🤗 Transformers `pipeline()` function to solve audio tasks with pre-trained models:

| # | Task | Section |
|---|------|---------|
| 1 | Audio classification | §1 |
| 2 | Automatic speech recognition (ASR) | §2 |
| 3 | Hands-on: transcribe a streamed VoxPopuli clip | §3 |
| 4 | Audio generation (TTS + music) — **optional** | §4 |

> **First run downloads models** to the Hugging Face cache (not the repo): the classifier
> (~1.2 GB) and the default ASR model (~360 MB). Everything runs on CPU.
> The optional generation models (Bark, MusicGen) are multi-GB — see §4.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from datasets import load_dataset, Audio
from transformers import pipeline

def load_minds14(name="en-AU", split="train"):
    kwargs = dict(path="PolyAI/minds14", name=name, split=split)
    try:
        return load_dataset(**kwargs)
    except Exception:
        return load_dataset(**kwargs, trust_remote_code=True)

print("ready")

## What is a `pipeline`?
A 🤗 Transformers **pipeline** bundles everything needed for a task — audio preprocessing, the
model, and turning raw model outputs into friendly results — behind a single call. You pick a
task string (e.g. `"audio-classification"`) and optionally a model, then call it on your audio.

## 1. Audio classification
We use a model fine-tuned on **MINDS-14** to predict the caller's *intent*. It expects **16 kHz**
audio, so we resample the dataset first.

In [ ]:
minds = load_minds14().cast_column("audio", Audio(sampling_rate=16_000))
classifier = pipeline("audio-classification", model="anton-l/xtreme_s_xlsr_300m_minds14")

example = minds[0]
id2label = minds.features["intent_class"].int2str
true_label = id2label(example["intent_class"])
preds = classifier(example["audio"]["array"], top_k=5)
print("true intent:", true_label)
preds

In [ ]:
labels = [p["label"] for p in preds][::-1]
scores = [p["score"] for p in preds][::-1]
colors = ["tab:green" if l == true_label else "tab:blue" for l in labels]
plt.figure(figsize=(10, 4))
plt.barh(labels, scores, color=colors)
plt.xlim(0, 1); plt.xlabel("score")
plt.title(f"Top-5 predicted intents (true: {true_label})")
plt.show()

In [ ]:
# listen to the clip we just classified
ipd.Audio(example["audio"]["array"], rate=example["audio"]["sampling_rate"])

## 2. Automatic speech recognition
The default English ASR pipeline transcribes speech to text.

In [ ]:
asr = pipeline("automatic-speech-recognition")
result = asr(example["audio"]["array"])
print("prediction:", result["text"])
print("reference :", example.get("english_transcription"))

Note: `facebook/wav2vec2-base-960h` returns **UPPERCASE, no punctuation** (a LibriSpeech CTC
model) — that is expected, not a bug. For another language you would pass a matching model, e.g.
`pipeline("automatic-speech-recognition", model="maxidl/wav2vec2-large-xlsr-german")`.

## 3. Hands-on: transcribe a streamed VoxPopuli example
Stream a single example from **VoxPopuli** (no full download), transcribe it with the ASR
pipeline, and compare to the reference transcription.

In [ ]:
def load_voxpopuli_stream(lang="en"):
    kwargs = dict(path="facebook/voxpopuli", name=lang, split="train", streaming=True)
    try:
        return load_dataset(**kwargs)
    except Exception:
        return load_dataset(**kwargs, trust_remote_code=True)

try:
    vp = load_voxpopuli_stream("en")
    ex = next(iter(vp))
    print("sampling_rate:", ex["audio"]["sampling_rate"])
    print("pipeline text:", asr(ex["audio"]["array"])["text"])
    print("reference    :", ex.get("normalized_text") or ex.get("raw_text"))
except Exception as e:
    print("Could not stream VoxPopuli:", type(e).__name__, str(e).splitlines()[0][:120])
    print("(It may require accepting terms / huggingface-cli login.)")

## 4. Audio generation (optional)
Unit 2 also covers **generating** audio with a pipeline: text-to-speech (Bark) and music
(MusicGen). These models are multi-GB and **slow on CPU** (tens of seconds to minutes per clip),
so they are opt-in:

1. Install the extra: `uv sync --extra generation`
2. Set `RUN_GENERATION = True` in the next cell and re-run the generation cells.

With `RUN_GENERATION = False` (the default) the cells below do nothing, so *Run All* stays fast.

In [ ]:
RUN_GENERATION = False  # set True after: uv sync --extra generation

In [ ]:
# Text-to-speech with Bark
if RUN_GENERATION:
    tts = pipeline("text-to-speech", model="suno/bark-small")
    out = tts("Hello! This speech was generated on a CPU with the Bark small model.")
    display(ipd.Audio(out["audio"].squeeze(), rate=out["sampling_rate"]))
else:
    print("skipped (set RUN_GENERATION = True and `uv sync --extra generation`)")

In [ ]:
# Music generation with MusicGen
if RUN_GENERATION:
    music = pipeline("text-to-audio", model="facebook/musicgen-small")
    out = music("90s rock song with electric guitar and heavy drums",
                forward_params={"max_new_tokens": 256})
    display(ipd.Audio(out["audio"][0], rate=out["sampling_rate"]))
else:
    print("skipped (set RUN_GENERATION = True and `uv sync --extra generation`)")

---
🎉 **That's Unit 2.** You used pre-trained pipelines for audio classification, ASR, and audio
generation, and transcribed streamed audio. Next up: Unit 3, transformer architectures for audio.